# Study 9 v2: resume C-M (16x128) from step 54k on Kaggle
Settings (right panel): **Accelerator = GPU T4 x2 or P100**, **Internet = On**, add the private dataset `ecg-data` (bundle + `last.pt`).
Best: **Save Version -> Save & Run All (Commit)**. It runs in the background (up to 12 h) even if you close the browser, and keeps `/kaggle/working` as output.

In [ ]:
# 1. GPU check
!nvidia-smi

In [ ]:
# 2. Set up the working copy from the dataset (handles Kaggle auto-extracting the zip or not)
import os, glob, shutil, zipfile
WORK = '/kaggle/working/ECG'
inp = [p for p in glob.glob('/kaggle/input/**', recursive=True)]
zips = [p for p in inp if p.endswith('ecg_colab_bundle.zip')]
ckpt = [p for p in inp if os.path.basename(p) == 'last.pt']
print('zip:', zips, '\ncheckpoint:', ckpt)
if not os.path.exists(WORK + '/latent_diffusion/train_diffusion_v2.py'):
    if zips:
        zipfile.ZipFile(zips[0]).extractall(WORK)
    else:   # Kaggle extracted it: find the folder that contains latent_diffusion/
        src = [os.path.dirname(os.path.dirname(p)) for p in inp if p.endswith('latent_diffusion/train_diffusion_v2.py')][0]
        shutil.copytree(src, WORK, dirs_exist_ok=True)
assert len(ckpt) == 1, 'expected exactly one last.pt in the dataset'
dst = WORK + '/latent_diffusion/artifacts/v2/diffusion_cm/last.pt'
os.makedirs(os.path.dirname(dst), exist_ok=True)
if not os.path.exists(dst):
    shutil.copy(ckpt[0], dst)
os.chdir(WORK)
!ls; ls -la latent_diffusion/artifacts/v2/diffusion_cm/

In [ ]:
# 3. Libraries. The pykeops test runs in a separate process so this notebook never holds the GPU.
!pip -q install einops opt_einsum pykeops
!python -c "import pykeops; pykeops.test_torch_bindings(); print('pykeops OK')" || echo 'PYKEOPS FAILED: STOP, the C-M checkpoint needs the conjugate-pair kernel'

In [ ]:
# 4. Check the checkpoint is the 54k C-M one before training
!python -c "import torch; c=torch.load('latent_diffusion/artifacts/v2/diffusion_cm/last.pt',map_location='cpu',weights_only=False); print('step',c['step'],'config',c['config'])"


In [ ]:
# 5. RESUME TRAINING to 100k. The first progress line must read 'cm 55000 ...'.
import os
os.chdir('/kaggle/working/ECG')
!V2_SINGLE_GPU_HOST=1 python -u -m latent_diffusion.train_diffusion_v2 --name cm --layers 16 --width 128 --state 64 --steps 100000 2>&1 | grep --line-buffered -v "pyKeOps\] Warning" 

In [ ]:
# 6. Pack results -> /kaggle/working/cm_results_kaggle.zip (download it from the Output tab)
import zipfile, glob, os
os.chdir('/kaggle/working/ECG')
files = (glob.glob('latent_diffusion/artifacts/v2/diffusion_cm/step*.pt') + ['latent_diffusion/artifacts/v2/diffusion_cm/last.pt']
         + glob.glob('latent_diffusion/results/v2/diffusion_training_cm.json') + glob.glob('latent_diffusion/audits/v2/phaseCD_cm*'))
with zipfile.ZipFile('/kaggle/working/cm_results_kaggle.zip', 'w', zipfile.ZIP_STORED) as z:
    for f in files:
        z.write(f); print(f)
!nvidia-smi --query-gpu=name --format=csv > /kaggle/working/cm_gpu.txt; cat /kaggle/working/cm_gpu.txt
# free space: keep only the zip in output
import shutil; shutil.rmtree('/kaggle/working/ECG')